# Previsão de Consumo de Massa Total

Vou usar as colunas `Qtd_Produzida`, `Qtd_Refugada` e `Qtd_Retrabalhada` para prever a coluna `Consumo_Massa_Total` usando os seguintes algoritmos de regressão:

* Decision Tree Regressor
* Random Forest Regressor
* XGBoost Regressor

Vou avaliar o desempenho de cada modelo usando as métricas:

* **MSE (Mean Squared Error)**
* **R-squared**

In [10]:
import pandas as pd
import altair as alt

In [11]:
# Load the dataset
df = pd.read_csv("../../Fase02/data/manutencao/dados_manutencao.csv")

# Rename columns
df = df.rename(
    columns={
        "Qt. Total Acumulada Produzida até a data específica": "Qtd_Produzida",
        "Qt. Acumulada Refugada até a data específica": "Qtd_Refugada",
        "Qtd. Acumulada total Retrabalhada até a data específica": "Qtd_Retrabalhada",
        "Consumo total de Massa Acumulada": "Consumo_Massa_Total",
    }
)

# Drop unnecessary columns
df = df.drop(
    [
        "Data de Produção Acumulada",
        "Cod. Ordem",
        "Cod Recurso",
        "Cod Produto",
        "Fator Un.",
        "Cód. Un.",
        "Descrição da massa (Composto)",
        "Tempo Restante para Manutenção",
    ],
    axis=1,
)

# Split into features and target
X = df.drop("Consumo_Massa_Total", axis=1)
y = df["Consumo_Massa_Total"]

In [12]:
# Print the first 5 rows of the dataframe
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

# Print the column name and their data types
print(df.info())

| Qtd_Produzida   | Qtd_Refugada   | Qtd_Retrabalhada   | Consumo_Massa_Total   |
|:----------------|:---------------|:-------------------|:----------------------|
| 87              | 98             | 14                 | 0.909355              |
| 819             | 1              | 8                  | 0.796544              |
| 84              | 74             | 4                  | 0.686085              |
| 868             | 19             | 47                 | 1.22212               |
| 95              | 69             | 25                 | 0.753044              |
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Qtd_Produzida        1000 non-null   int64  
 1   Qtd_Refugada         1000 non-null   int64  
 2   Qtd_Retrabalhada     1000 non-null   int64  
 3   Consumo_Massa_Total  1000 non-null   float64
dtypes: float64(1), i

In [14]:
# Create a scatter plot for each column
scatter_produzida = (
    alt.Chart(df)
    .mark_point()
    .encode(
        x=alt.X("Qtd_Produzida", title="Qtd_Produzida"),
        y=alt.Y("Consumo_Massa_Total", title="Consumo_Massa_Total"),
        tooltip=["Qtd_Produzida", "Consumo_Massa_Total"],
    )
    .properties(title="Qtd_Produzida vs Consumo_Massa_Total")
    .interactive()
)

scatter_refugada = (
    alt.Chart(df)
    .mark_point()
    .encode(
        x=alt.X("Qtd_Refugada", title="Qtd_Refugada"),
        y=alt.Y("Consumo_Massa_Total", title="Consumo_Massa_Total"),
        tooltip=["Qtd_Refugada", "Consumo_Massa_Total"],
    )
    .properties(title="Qtd_Refugada vs Consumo_Massa_Total")
    .interactive()
)

scatter_retrabalhada = (
    alt.Chart(df)
    .mark_point()
    .encode(
        x=alt.X("Qtd_Retrabalhada", title="Qtd_Retrabalhada"),
        y=alt.Y("Consumo_Massa_Total", title="Consumo_Massa_Total"),
        tooltip=["Qtd_Retrabalhada", "Consumo_Massa_Total"],
    )
    .properties(title="Qtd_Retrabalhada vs Consumo_Massa_Total")
    .interactive()
)

# Combine the plots into a single figure
chart = scatter_produzida | scatter_refugada | scatter_retrabalhada
path_class = "classification_images/"
# Save the chart
chart.save(path_class + "scatter_plots_03.json")
chart.save(path_class + "scatter_plots_03.png")

In [15]:
# Importar pacotes para análise de dados
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor  # Import from sklearn.tree
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

In [16]:
# Load the dataset
df = pd.read_csv("../../Fase02/data/manutencao/dados_manutencao.csv")

# Rename columns
df = df.rename(
    columns={
        "Qt. Total Acumulada Produzida até a data específica": "Qtd_Produzida",
        "Qt. Acumulada Refugada até a data específica": "Qtd_Refugada",
        "Qtd. Acumulada total Retrabalhada até a data específica": "Qtd_Retrabalhada",
        "Consumo total de Massa Acumulada": "Consumo_Massa_Total",
    }
)

# Drop unnecessary columns
df = df.drop(
    [
        "Data de Produção Acumulada",
        "Cod. Ordem",
        "Cod Recurso",
        "Cod Produto",
        "Fator Un.",
        "Cód. Un.",
        "Descrição da massa (Composto)",
        "Tempo Restante para Manutenção",
    ],
    axis=1,
)

# Split into features and target
X = df.drop("Consumo_Massa_Total", axis=1)
y = df["Consumo_Massa_Total"]

In [17]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train and evaluate models
models = {
    "DecisionTreeRegressor": DecisionTreeRegressor(),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=100),
    "XGBRegressor": xgb.XGBRegressor(),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{name}:")
    print(f"  Mean Squared Error: {mse:.2f}")
    print(f"  R-squared: {r2:.2f}")

DecisionTreeRegressor:
  Mean Squared Error: 0.16
  R-squared: -0.83
RandomForestRegressor:
  Mean Squared Error: 0.10
  R-squared: -0.17
XGBRegressor:
  Mean Squared Error: 0.12
  R-squared: -0.41


# Previsão de Consumo de Massa Total

Vou usar as colunas `Qtd_Produzida`, `Qtd_Refugada` e `Qtd_Retrabalhada` para prever a coluna `Consumo_Massa_Total` usando os seguintes algoritmos de regressão:

* Decision Tree Regressor
* Random Forest Regressor
* Gradient Boosting Regressor

Vou avaliar o desempenho de cada modelo usando as métricas:

* **MSE (Mean Squared Error)**
* **R-squared**

In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load the dataset
df = pd.read_csv("../../Fase02/data/manutencao/dados_manutencao.csv")

# Rename columns
df = df.rename(
    columns={
        "Qt. Total Acumulada Produzida até a data específica": "Qtd_Produzida",
        "Qt. Acumulada Refugada até a data específica": "Qtd_Refugada",
        "Qtd. Acumulada total Retrabalhada até a data específica": "Qtd_Retrabalhada",
        "Consumo total de Massa Acumulada": "Consumo_Massa_Total",
    }
)

# Drop unnecessary columns
df = df.drop(
    [
        "Data de Produção Acumulada",
        "Cod. Ordem",
        "Cod Recurso",
        "Cod Produto",
        "Fator Un.",
        "Cód. Un.",
        "Descrição da massa (Composto)",
        "Tempo Restante para Manutenção",
    ],
    axis=1,
)

# Split into features and target
X = df.drop("Consumo_Massa_Total", axis=1)
y = df["Consumo_Massa_Total"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train and evaluate models
models = {
    "DecisionTreeRegressor": DecisionTreeRegressor(),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=100),
    "GradientBoostingRegressor": GradientBoostingRegressor(),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{name}:")
    print(f"  Mean Squared Error: {mse:.2f}")
    print(f"  R-squared: {r2:.2f}")

DecisionTreeRegressor:
  Mean Squared Error: 0.16
  R-squared: -0.88
RandomForestRegressor:
  Mean Squared Error: 0.10
  R-squared: -0.17
GradientBoostingRegressor:
  Mean Squared Error: 0.10
  R-squared: -0.10


## Análise dos Resultados dos Modelos de Regressão

Com base nos resultados, podemos observar que os modelos baseados em árvore (DecisionTreeRegressor, RandomForestRegressor e GradientBoostingRegressor) têm um MSE maior e um R-quadrado negativo, sugerindo que eles têm um desempenho ruim nos dados. É necessário a utilizacao dos dados reais solicitados a SABO.

### Observações:

* O conjunto de dados pode não ter uma relação **não linear** forte entre as variáveis preditoras e a variável alvo, o que pode explicar o baixo desempenho dos modelos baseados em árvore.
* Os modelos baseados em árvore podem estar sofrendo de **overfitting**, levando a um desempenho ruim nos dados de teste.
* Ajustar os **hiperparâmetros** dos modelos ou usar algoritmos de regressão mais complexos pode melhorar o desempenho do modelo.